In [65]:
import sys
from pathlib import Path

# Get the absolute path of the parent directory of your current working directory
ROOT = Path.cwd().resolve().parent
# Add this parent directory to the top of Python's module search path
sys.path.insert(0, str(ROOT))


In [75]:
import re
import shutil
import pandas as pd
from pathlib import Path
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"C:\Users\manja\Downloads\0726_FECFIN_KMC.xlsx")
origem_lanc = Path(r"C:\Users\manja\Documents\auto_extrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"C:\Users\manja\Downloads")

df = pd.read_excel(arquivo, sheet_name="07-2026")

display(df)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,NaN,NaN,A. MONTEIRO CONSTRUTORA,NaN,NaN,2025-01-01 00:00:00,OBRA,DESCRIÇÃO,NF,CATEGORIA,...,BANCO,ENTRADA,SAÍDA,DATA,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,PLANO DE CONTAS,NaN,NaN,NaN,Saldo anterior ->,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15068.47,NaN,13.61
2,NaN,NaN,NaN,NaN,NaN,NaN,-,Honorarios contábeis - KMC,NaN,Contador,...,Sicoob - KMC,NaN,1275.41,2026-07-01 00:00:00,Reforma Kenia,1.1,NaN,13793.06,NaN,13.61
3,NaN,NaN,1,VENDAS TOTAIS,92514.79,NaN,-,Limpeza do escritório,NaN,Limpeza do escritório (diarista),...,Sicoob - KMC,NaN,360,2026-07-01 00:00:00,Obra - 178 - Estacionamento T-35,1.2,NaN,13433.06,NaN,13.61
4,NaN,NaN,1.1,Reforma Kenia,0.00,NaN,Obra - 170 - Acompanhamento Fazenda,Prestação de serviços técnicos de engenharia p...,91,Avilages,...,Sicoob - KMC,4000,NaN,2026-07-01 00:00:00,Obra - 178 - Estacionamento T-35 - nota fiscal,1.3,NaN,17433.06,NaN,13.61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RECEITAS NÃO OPERACIONAIS,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DESPESAS NÃO OPERACIONAIS,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INVESTIMENTOS,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RETIRADA DE LUCROS,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
df.columns = df.iloc[0] # Definindo nomes das colunas com base na primeira linha
df = df[2:].reset_index(drop=True) # Removendo as duas primeiras linhas, que agora são redundantes
df = df[["OBRA", "DESCRIÇÃO", "NF", "CATEGORIA", "BANCO", "ENTRADA", "SAÍDA", "DATA"]] # Selecionando apenas as colunas desejadas
df["DATA"] = pd.to_datetime(df["DATA"], format="%d/%m/%Y") # Convertendo a coluna "DATA" para o tipo datetime
df["VALOR"] = df.apply(lambda row: row["ENTRADA"] if pd.notnull(row["ENTRADA"]) else -row["SAÍDA"], axis=1) # Criando a coluna "VALOR" com base nas colunas "ENTRADA" e "SAÍDA"
df["NF"] = df["NF"].apply(lambda x: re.sub(r"\D", "", str(x)) if pd.notnull(x) else 0) # Removendo caracteres não numéricos da coluna "NF"
df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBRA']} {x['NF']} {x['DESCRIÇÃO']}", axis=1)
df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("-", "", regex=True).str.strip().str.replace("|", "", regex=True).str.replace("nan", "", regex=True).str.replace("0", "", regex=True).str.upper()
df["TIPO"] = df["VALOR"].apply(lambda x: "C" if x > 0 else "D") # Criando a coluna "TIPO" com base na coluna "VALOR"
banco = df["BANCO"].unique()[0] # Obtendo o nome do banco a partir da coluna "BANCO"
banco = banco.split(" - ")[0].upper() # Removendo o sufixo após o hífen e convertendo para maiúsculas
df = df[df["BANCO"].notna() & (df["BANCO"] != "")]
df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]].copy() # Selecionando apenas as colunas desejadas
df["VALOR"] = df["VALOR"].abs()
if not df.empty:
    arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{banco}.xlsm"
    shutil.copy2(origem_lanc, arquivo_lanc)
    planilha_lancamento(df, arquivo_lanc)

# display(df)

Arquivo preenchido com sucesso: C:\Users\manja\Downloads\[LANC] 0726_FECFIN_KMC_SICOOB.xlsm


In [59]:
import shutil
import re
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"C:\Users\manja\Downloads\0726_FECFIN_KMC.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

wb = load_workbook(arquivo, read_only=True, data_only=True)
abas = [
    ws.title
    for ws in wb.worksheets
    if ws.sheet_state == "visible"
]

if "ACR" in arquivo.stem:
    bancos = ["SICOOB", "CAIXA"]
    extratos = [aba for aba in abas for banco in bancos if banco in aba]
    for extrato in extratos:
        print(f"Processando a planilha '{extrato}' do arquivo '{arquivo.name}'...")
        df = pd.read_excel(arquivo, sheet_name=extrato)

        if "SICOOB" in extrato:
            df.columns = df.iloc[1] # Definindo cabeçalho das colunas
            df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS / CONT']} {x['OBS / INT']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
        elif "CAIXA" in extrato:
            df.columns = df.iloc[4] # Definindo cabeçalho das colunas
            df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']}", axis=1)

            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.replace("None", "", regex=False).str.strip().str.upper()
            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].replace("", pd.NA)
            df = df.dropna(subset=["DESCRIÇÃO"])

        if df.empty:
            print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
            continue

        df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
        df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
        df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
        df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
        df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
        df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
        df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
        df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
        df["VALOR"] = df["VALOR"].abs()
        df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

        if not df.empty:
            arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
            shutil.copy2(origem_lanc, arquivo_lanc)
            planilha_lancamento(df, arquivo_lanc)

display(df)

ModuleNotFoundError: No module named 'pypdf'

In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_ACR.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

wb = load_workbook(arquivo, read_only=True, data_only=True)
abas = [
    ws.title
    for ws in wb.worksheets
    if ws.sheet_state == "visible"
]

print(abas)

if "CEMAF 60" in arquivo.stem:
    bancos = ["SICOOB", "CAIXA"]
    extratos = [aba for aba in abas for banco in bancos if banco in aba]
    for extrato in extratos:
            print(f"Processando a planilha '{extrato}' do arquivo '{arquivo.name}'...")
            df = pd.read_excel(arquivo, sheet_name=extrato)

            if "SICOOB" in extrato:
                df.columns = df.iloc[1] # Definindo cabeçalho das colunas
                df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['OBS / INT']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
            elif "CAIXA" in extrato:
                df.columns = df.iloc[4] # Definindo cabeçalho das colunas
                df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']}", axis=1)

                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.replace("None", "", regex=False).str.strip().str.upper()
                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].replace("", pd.NA)
                df = df.dropna(subset=["DESCRIÇÃO"])

            if df.empty:
                print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
                continue

            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
            df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
            df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
            df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
            df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
            df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
            df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
            df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
            df["VALOR"] = df["VALOR"].abs()
            df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

            if not df.empty:
                arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
                shutil.copy2(origem_lanc, arquivo_lanc)
                planilha_lancamento(df, arquivo_lanc)

display(df)

['RESUMO', 'NFS FORNEC', 'FATXREC', 'SICOOB - 15619-1', 'CAIXA', 'ADIANT A FORNECEDOR', 'INFORMAÇÕES DIVERSAS ']
Processando a planilha 'SICOOB - 15619-1' do arquivo '0626_FECFIN_CEMAF 60.xlsx'...
Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\[LANC] 0626_FECFIN_CEMAF 60_SICOOB - 15619-1.xlsm
Processando a planilha 'CAIXA' do arquivo '0626_FECFIN_CEMAF 60.xlsx'...


4,DATA,DESCRIÇÃO,VALOR,TIPO


In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_ADP PART.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

wb = load_workbook(arquivo, read_only=True, data_only=True)
abas = [
    ws.title
    for ws in wb.worksheets
    if ws.sheet_state == "visible"
]

if "ALOHA" in arquivo.stem:
    bancos = ["INTER", "CAIXA"]
    extratos = [aba for aba in abas for banco in bancos if banco in aba]

    for extrato in extratos:
        print(f"Processando a planilha '{extrato}' do arquivo '{arquivo.name}'...")
        df = pd.read_excel(arquivo, sheet_name=extrato)

        if "INTER" in extrato:
            df.columns = df.iloc[1] # Definindo cabeçalho das colunas
            df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['OBS INTERNA']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
        elif "CAIXA" in extrato:
            df.columns = df.iloc[4] # Definindo cabeçalho das colunas
            df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']} {x['OBS INT']}", axis=1)

        if df.empty:
            print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
            continue

        df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
        df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
        df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
        df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
        df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
        df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
        df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
        df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
        df["VALOR"] = df["VALOR"].abs()
        df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

        if not df.empty:
            arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
            shutil.copy2(origem_lanc, arquivo_lanc)
            planilha_lancamento(df, arquivo_lanc)

    display(df)

In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_CE PART.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

with pd.ExcelFile(arquivo) as xls:
    abas = xls.sheet_names

    if "CE PART" in arquivo.stem:
        bancos = ["CORA", "CAIXA"]
        extratos = [aba for aba in abas for banco in bancos if banco in aba]

        for extrato in extratos:
            df = pd.read_excel(arquivo, sheet_name=extrato)

            if "CORA" in extrato:
                df.columns = df.iloc[1] # Definindo cabeçalho das colunas
                df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['OBS INTERNA']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
            elif "CAIXA" in extrato:
                df.columns = df.iloc[4] # Definindo cabeçalho das colunas
                df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']} {x['OBS INT']}", axis=1)
            
            if df.empty:
                print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
                continue

            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
            df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
            df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
            df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
            df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
            df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
            df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
            df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
            df["VALOR"] = df["VALOR"].abs()
            df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

            if not df.empty:
                arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
                shutil.copy2(origem_lanc, arquivo_lanc)
                planilha_lancamento(df, arquivo_lanc)